# C 扩展

学习目标：读懂一个小型 Python/C API 扩展，正确处理整数边界、错误返回和引用所有权，并完成本机构建、临时安装与行为检查。

前置知识：函数与异常、元组与对象引用、模块导入、弱引用、文件路径、上下文管理器、子进程和 wheel 打包；本章补充示例所需的 C 语法。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本例面向 Windows x64，需 MSVC C 编译工具、Windows SDK，以及当前 Python 的头文件与导入库。

构建与测试工具版本见 [requirements.txt](requirements.txt)。源码副本、构建和安装目标均放在临时目录，扩展子进程退出后清理。

配套脚本：位于 [scripts/34-c-extensions/](scripts/34-c-extensions/)。

（1）[study\_c\_api.c](scripts/34-c-extensions/study_c_api.c)：两个扩展函数、方法表和模块初始化。

（2）[pyproject.toml](scripts/34-c-extensions/pyproject.toml) 与 [setup.py](scripts/34-c-extensions/setup.py)：项目元数据、构建依赖和扩展编译声明。

（3）[tests/test\_extension.py](scripts/34-c-extensions/tests/test_extension.py)：参数边界、异常传播、对象身份和引用释放测试。

（4）[README.md](scripts/34-c-extensions/README.md)：随源码分发的简短用途与运行条件。

## 1 C 局部变量、指针与 Python 对象

C 扩展把编译后的函数接入 CPython。本例导入名为 study\_c\_api：add\_nonnegative 相加两个非负整数，tuple\_item 取出元组中的一个对象。它用于观察接口与引用管理，不以速度提升为目标；Python/C API 的这些约定属于 CPython。

| 写法 | 中文名称／含义 |
| --- | --- |
| long left | 声明一个 C 有符号长整数变量 left，范围有限 |
| PyObject \*p | 声明指针 p，保存 Python 对象的地址 |
| &left | 取得变量 left 的地址，供参数解析函数写入转换结果 |
| \*p | 解引用，访问 p 指向的对象；前提是指针有效 |
| NULL | 空指针值，不指向一个可访问的 Python 对象 |

局部变量通常由调用栈（stack）承载，具体位置也可能受编译器优化影响。指针变量的局部生命周期，与它指向的对象生命周期不同：Python 对象由解释器的私有堆（heap）管理。函数返回对象指针时，必须保证调用者取得有效引用，不能返回已失效的局部变量地址。

Python.h 提供 API 声明，应先于其他头文件包含。Python 3.12 的扩展文档建议先定义 PY\_SSIZE\_T\_CLEAN；本例保留该写法，整数上界由 limits.h 提供。下面只读取真实源码，后面会编译它。

In [1]:
from pathlib import Path

project_dir = Path("scripts/34-c-extensions").resolve()
c_source = (project_dir / "study_c_api.c").read_text(encoding="utf-8")
parse_start = c_source.index("    if (!PyArg_ParseTuple")
print(c_source[:parse_start].strip())
# PyObject * 是返回值和参数的指针类型；left、right 是局部 C long。
# (void)self 表明本函数没有使用传入的模块对象。

/*
所属章节：34-C 扩展
演示知识点：CPython 3.12 教学扩展，非负整数加法（溢出前检查）与元组元素引用（借用转强引用）、方法表与模块初始化，错误路径保留 Python 异常
运行命令：PYTHONPATH=<扩展安装目录> python -c "import study_c_api; print(study_c_api.add_nonnegative(20, 55), study_c_api.tuple_item(('C', 'Python'), 1))"（工作目录 content/编程语言/python）
期望结果：输出 75 Python；<扩展安装目录> 为 pip --target 安装本章构建 wheel 的目录
*/
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <limits.h>

/* 参数解析与有界算术。 */
static PyObject *
add_nonnegative(PyObject *self, PyObject *args)
{
    long left;
    long right;
    (void)self;


## 2 参数解析、算术边界与错误返回

PyArg\_ParseTuple 按格式字符串从实参元组提取值。格式必须与后续 C 变量地址的类型一致，地址写错可能破坏内存，不能依赖 Python 的类型检查补救。

| 格式 | 中文名称／含义 | 对应的 C 存储位置 |
| --- | --- | --- |
| l | 有符号长整数转换，超出范围报 OverflowError | long 变量的地址 |
| n | 可用于长度和索引的有符号整数转换 | Py\_ssize\_t 变量的地址 |
| O | 保存原对象指针，不做类型转换，取得借用引用 | PyObject \* 变量的地址 |
| O! | 检查指定 Python 类型后保存对象指针 | 类型对象地址，再传对象指针变量的地址 |

ll:add\_nonnegative 表示两个 l 参数，冒号后的名称用于错误信息。l 和 n 支持整数索引协议，也就是对象的 \_\_index\_\_ 方法；本例接受 bool，True 和 False 分别按 1 和 0 处理，不接受 float 或只有 \_\_int\_\_ 的对象。

left 和 right 表示转换后的两个加数。先拒绝负数，再检查 left > LONG\_MAX - right；这里 LONG\_MAX 是当前 C long 上界。通过后才计算 left + right，避免先发生有符号整数溢出。Python int 能表达很大的整数，并不表示转换后的 C long 也能。

解析失败时 API 已设置异常，直接返回 NULL。业务检查失败时先调用 PyErr\_SetString 再返回 NULL。成功由 PyLong\_FromLong 返回新的 Python 整数引用；若它分配失败，也会设置异常并返回 NULL。NULL 表示失败，与真正的 Python 对象 None 不同。

In [2]:
tuple_start = c_source.index("/* 从借用引用")
print(c_source[parse_start:tuple_start].strip())
# 能转为 long 但为负数时，业务分支报 ValueError。
# 大到不能转为 long 的值，会更早在参数解析阶段报 OverflowError。
# 判断只在非负值上计算 LONG_MAX - right，因此该减法也在范围内。

if (!PyArg_ParseTuple(args, "ll:add_nonnegative", &left, &right)) {
        return NULL;
    }
    if (left < 0 || right < 0) {
        PyErr_SetString(PyExc_ValueError, "values must be nonnegative");
        return NULL;
    }
    /* 先比较余量，不能先计算可能溢出的 left + right。 */
    if (left > LONG_MAX - right) {
        PyErr_SetString(PyExc_OverflowError, "sum exceeds C long");
        return NULL;
    }
    return PyLong_FromLong(left + right);
}


## 3 借用引用与新引用

引用所有权说明谁负责保持对象存活、谁负责释放引用，不表示谁拥有对象的一份拷贝。

| 原文名称 | 中文名称／含义 | 调用者的责任 |
| --- | --- | --- |
| borrowed reference | 借用引用，依靠其他所有者保持对象存活 | 不为这次借用调用 Py\_DECREF |
| new reference | 新引用，调用者取得一个强引用 | 用完释放，或明确转交给调用者等其他所有者 |
| stolen reference | 被窃取的引用，接收方接管传入引用 | 按该 API 的成功和失败约定处理，不能重复释放 |

tuple\_item 使用 O!n：先要求元组，再读取索引。PyTuple\_GetItem 返回借用引用，负数和越界索引都会设置 IndexError 并返回 NULL；这里没有 Python 下标语法中的负索引换算。

实参元组保持 items 存活，items 又保持元素存活。检查 item 不是 NULL 后，Py\_NewRef 为同一对象取得新强引用，再作为函数结果返回。它不复制对象，也不保证增加某个可观察到的计数值；例如 CPython 3.12 的不朽对象有特殊计数行为。

作为对照，PyTuple\_SetItem 会窃取传入元素的引用，只应操作自己正在构造的元组。本例不需要修改元组，不使用它。

In [3]:
methods_start = c_source.index("/* 将 Python 名称")
print(c_source[tuple_start:methods_start].strip())
# items 与 item 都没有取得独立所有权，不能对它们随意 DECREF。
# Py_NewRef 的结果转交给 Python 调用者；返回前再 DECREF 会抵消它。

/* 从借用引用取得可返回的新强引用。 */
static PyObject *
tuple_item(PyObject *self, PyObject *args)
{
    PyObject *items;
    PyObject *item;
    Py_ssize_t index;
    (void)self;

    if (!PyArg_ParseTuple(args, "O!n:tuple_item",
                          &PyTuple_Type, &items, &index)) {
        return NULL;
    }
    item = PyTuple_GetItem(items, index);
    if (item == NULL) {
        return NULL;
    }
    /* items 与 item 都是借用引用；本函数不 DECREF 它们。 */
    return Py_NewRef(item);
}


## 4 方法表与模块初始化入口

| 名称 | 中文名称／含义 |
| --- | --- |
| PyMethodDef | 方法表的一条记录，关联 Python 名称、C 函数、调用约定和说明 |
| METH\_VARARGS | 把位置实参作为元组传给扩展函数，不接收关键字实参 |
| PyModuleDef | 模块定义，包含名称、状态设置和方法表等字段 |
| PyMODINIT\_FUNC | 声明扩展初始化函数所需的返回类型和平台导出方式 |
| PyInit\_name | 初始化入口命名形式，name 是导入模块名，本例为 study\_c\_api |

methods 是 C 结构体数组，最后的全空记录标记结束。函数前的 static 把这些辅助符号限制在当前 C 源文件内；初始化入口需要导出。self 对模块级函数指向模块对象，args 指向位置实参元组。

本例采用单阶段初始化（single-phase initialization），由入口创建并返回模块；m\_size 写为 -1，不声明子解释器支持。多阶段初始化（multi-phase initialization）把模块创建与执行初始化分开，需要另行设计模块状态和相关槽位，不能只修改一个数字完成迁移。

In [4]:
init_start = c_source.index("/* 初始化中")
print(c_source[methods_start:init_start].strip())
# 两个公开名称都对应 METH_VARARGS；下面的位置顺序对应 PyModuleDef。
# -1 是 m_size；后面的四个 NULL 不提供多阶段槽位与状态清理回调。

/* 将 Python 名称、C 函数与调用约定连接起来。 */
static PyMethodDef methods[] = {
    {"add_nonnegative", add_nonnegative, METH_VARARGS,
     "Add two nonnegative indexable integers within C long range."},
    {"tuple_item", tuple_item, METH_VARARGS,
     "Return the same tuple item at a nonnegative index."},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef module_definition = {
    PyModuleDef_HEAD_INIT,
    "study_c_api",
    "Minimal CPython 3.12 C API examples.",
    -1,
    methods,
    NULL,
    NULL,
    NULL,
    NULL
};


## 5 初始化失败时精确释放引用

初始化先创建模块，再把当前 LONG\_MAX 作为属性加入模块。PyModule\_Create 与 PyLong\_FromLong 成功时都返回新引用。PyModule\_AddObjectRef 为模块添加对象引用，不窃取调用者已有的引用，成功返回 0，失败返回 -1 并设置异常。

因此 limit 创建失败时，只释放已经创建的 module；属性添加结束后，无论成功与否都释放本地 limit 引用。添加失败再释放 module，成功则把 module 的新引用交给导入系统。每个出口只处理自己已经取得的所有权。

Py\_DECREF 要求非 NULL 指针；Py\_XDECREF 可接收 NULL。释放强引用可能触发对象析构乃至 Python 代码，所以释放前应把相关状态安排好，不能把它理解成无条件的“计数减一”。

In [5]:
print(c_source[init_start:].strip())
# limit == NULL 分支没有释放 limit，因为那里没有取得对象引用。
# AddObjectRef 之后统一 DECREF(limit)，不会因失败路径遗漏本地引用。
# 成功返回 module 后，导入系统接管该返回引用。

/* 初始化中各条退出路径都交代已取得的强引用。 */
PyMODINIT_FUNC
PyInit_study_c_api(void)
{
    PyObject *module = PyModule_Create(&module_definition);
    PyObject *limit;
    if (module == NULL) {
        return NULL;
    }

    limit = PyLong_FromLong(LONG_MAX);
    if (limit == NULL) {
        Py_DECREF(module);
        return NULL;
    }
    /* AddObjectRef 不窃取 limit；无论成功与否都释放本地所有权。 */
    int status = PyModule_AddObjectRef(module, "LONG_MAX", limit);
    Py_DECREF(limit);
    if (status < 0) {
        Py_DECREF(module);
        return NULL;
    }
    return module;
}


## 6 声明扩展构建配置

pyproject.toml 声明分发包 study-c-api-demo 和 setuptools.build\_meta 后端。setup.py 中的 setuptools.Extension 把导入名 study\_c\_api 与 C 源文件连接起来；这里没有 Python 包目录。

本例构建选项使用 MSVC 的 /utf-8，以 UTF-8 读取包含中文注释的源文件。requires-python 暂限定 >=3.12,<3.13，体现本例的验证范围；该字段是安装条件，本身不会证明二进制兼容。

构建前端会调用 setup.py 中的声明，不直接运行 setup.py install。编译器、Python.h、Python 导入库和 Windows SDK 必须事先存在；遇到缺失应先解决运行条件。

In [6]:
import tomllib

project_text = (project_dir / "pyproject.toml").read_text(encoding="utf-8")
configuration = tomllib.loads(project_text)
print(configuration["build-system"])  # setuptools 83.0.0、wheel 0.47.0；后端为 setuptools.build_meta。
print(configuration["project"]["requires-python"])  # >=3.12,<3.13。
print((project_dir / "setup.py").read_text(encoding="utf-8"))  # 显示构建声明：study_c_api.c 与 MSVC 的 /utf-8 参数。
# 构建依赖与已安装工具版本一致；模块名不是分发包名。

{'requires': ['setuptools==83.0.0', 'wheel==0.47.0'], 'build-backend': 'setuptools.build_meta'}
>=3.12,<3.13
"""所属章节：34-C 扩展
演示知识点：setuptools Extension 声明 study_c_api 扩展的源码与 /utf-8 编译参数
运行命令：python -m build --no-isolation --outdir <输出目录> scripts/34-c-extensions（工作目录 content/编程语言/python）
期望结果：生成含 study_c_api 扩展的 wheel 与 sdist；<输出目录> 为自选的构建产物目录
"""

import setuptools


def main() -> None:
    """构建后端调用此入口；不直接执行 setup.py 安装。"""
    setuptools.setup(
        ext_modules=[
            setuptools.Extension(
                "study_c_api",
                sources=["study_c_api.c"],
                extra_compile_args=["/utf-8"],
            )
        ]
    )


if __name__ == "__main__":
    main()



## 7 在临时源码副本中构建

python -m build 默认先构建 sdist，再从该源码分发包构建 wheel。这同时检查 C 源文件是否进入了 sdist。--no-isolation 让构建使用当前环境已有依赖，不创建隔离环境。

编译会产生中间文件，因此先把配套目录复制到 TemporaryDirectory。构建完成后只保留两个小归档的字节，退出 with 会删除副本。子进程环境从当前环境复制，关闭字节码写入，并将临时路径限制在这次工作目录内。

In [7]:
import os
import shutil
import subprocess
import sys
import tempfile

child_env = os.environ.copy()
# 只在子进程副本中移除继承的强制着色，避免与 NO_COLOR 同时生效。
child_env.pop("FORCE_COLOR", None)
child_env.update({
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
})
# 先复制源码，构建目录和中间文件随临时目录一起清理。
with tempfile.TemporaryDirectory() as directory:
    build_root = Path(directory)
    source_copy = shutil.copytree(project_dir, build_root / "source")
    artifacts = build_root / "artifacts"
    build_env = {**child_env, "TEMP": directory, "TMP": directory}
    built = subprocess.run(
        [sys.executable, "-B", "-m", "build", "--no-isolation",
         "--outdir", str(artifacts), str(source_copy)],
        cwd=build_root, env=build_env, capture_output=True,
        encoding="utf-8", check=True, timeout=180,
    )
    (wheel_path,) = artifacts.glob("*.whl")
    (sdist_path,) = artifacts.glob("*.tar.gz")
    wheel_name, sdist_name = wheel_path.name, sdist_path.name
    wheel_bytes, sdist_bytes = wheel_path.read_bytes(), sdist_path.read_bytes()
    print(wheel_name)  # 本章环境：study_c_api_demo-0.1.0-cp312-cp312-win_amd64.whl。
    print(sdist_name)  # study_c_api_demo-0.1.0.tar.gz。
    print(built.stderr, end="")  # 保留构建警告，失败由 check=True 直接传播。
assert not build_root.exists()
# 编译或链接失败会立即抛出异常，不能把输出文件名当作成功凭据。
# 保留实际警告；本共享 Conda 构建可能提示 Py_DEBUG 配置变量未设置。
# 关闭字节码也可能触发跳过 byte-compiling 的提示。

study_c_api_demo-0.1.0-cp312-cp312-win_amd64.whl
study_c_api_demo-0.1.0.tar.gz
* Getting build dependencies for sdist...
* Building sdist...
* Building wheel from sdist
* Getting build dependencies for wheel...
* Building wheel...

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\setuptools\command\bdist_wheel.py:103: RuntimeWarning: Config variable 'Py_DEBUG' is unset, Python ABI tag may be incorrect
  if get_flag("Py_DEBUG", hasattr(sys, "gettotalrefcount"), warn=(impl == "cp")):


## 8 检查真实分发内容

Windows 扩展的可导入二进制文件使用 .pyd 后缀。wheel 内应包含它及 .dist-info 元数据；sdist 则需要包含 C 源码、构建声明和项目说明。

下面使用 ZIP 与 tar 读取刚构建的归档，不向源码目录解压。元数据里的 Requires-Python 还需要与后面检查的 wheel 标签共同判断安装条件。

In [8]:
import email.parser
import email.policy
import io
import tarfile
import zipfile

# 先读 wheel 的二进制和元数据，再读 sdist 的源文件清单。
with zipfile.ZipFile(io.BytesIO(wheel_bytes)) as archive:
    wheel_members = archive.namelist()
    (binary_name,) = [name for name in wheel_members if name.endswith(".pyd")]
    (metadata_name,) = [
        name for name in wheel_members if name.endswith(".dist-info/METADATA")
    ]
    parser = email.parser.BytesParser(policy=email.policy.default)
    metadata = parser.parsebytes(archive.read(metadata_name))
    print(binary_name)  # 本章环境：study_c_api.cp312-win_amd64.pyd。
    print(metadata["Name"], metadata["Requires-Python"])  # study-c-api-demo <3.13,>=3.12。
    assert metadata["Name"] == "study-c-api-demo"
    assert metadata["Requires-Python"] == "<3.13,>=3.12"

with tarfile.open(fileobj=io.BytesIO(sdist_bytes), mode="r:gz") as archive:
    sdist_members = archive.getnames()
    prefix = sdist_name.removesuffix(".tar.gz")
    for name in ["study_c_api.c", "pyproject.toml", "setup.py", "README.md"]:
        assert f"{prefix}/{name}" in sdist_members
        print(name)  # 依次为 study_c_api.c、pyproject.toml、setup.py、README.md。
# C 源码进入 sdist，而 Windows 的可导入扩展进入 wheel。

study_c_api.cp312-win_amd64.pyd
study-c-api-demo <3.13,>=3.12
study_c_api.c
pyproject.toml
setup.py
README.md


## 9 临时安装与子进程导入

pip install --target 指定安装目录，--no-index 禁止查询包索引，--no-deps 不安装依赖。本例没有第三方运行依赖，只安装本地 wheel；--target 不会另建 Python 环境。

下面的 run\_installed 把这些操作集中起来，后续每个小实验都调用它。传入的 code 是本章自己编写的 Python 源码字符串，由当前解释器的子进程执行。工作目录离开源码目录，PYTHONPATH 仅在子进程中指向安装位置。

子进程结束后再退出临时目录，避免 Notebook 内核持续占用已加载的 .pyd 文件。安装失败或实验失败直接抛出 CalledProcessError，诊断保存在异常的 stdout 和 stderr 中；临时目录由 with 清理。

In [9]:
def run_installed(code: str) -> str:
    """临时安装本章 wheel，执行一次子进程实验并返回真实输出。"""
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        local_wheel = root / wheel_name
        local_wheel.write_bytes(wheel_bytes)
        target = root / "installed"
        environment = {
            **child_env, "TEMP": directory, "TMP": directory,
            "PYTHONPATH": str(target), "STUDY_INSTALL_TARGET": str(target),
        }
        # 1. 安装仅发生在本次临时目录，不改共享环境的 site-packages。
        installed = subprocess.run(
            [sys.executable, "-B", "-m", "pip", "install", str(local_wheel),
             "--target", str(target), "--no-index", "--no-deps",
             "--no-compile", "--no-cache-dir", "--disable-pip-version-check"],
            cwd=root, env=environment, capture_output=True,
            encoding="utf-8", check=True, timeout=60,
        )
        # 2. 只有子进程导入扩展，退出后才清理已安装的二进制文件。
        result = subprocess.run(
            [sys.executable, "-B", "-c", code],
            cwd=root, env=environment, capture_output=True,
            encoding="utf-8", check=True, timeout=60,
        )
    assert not root.exists()
    return result.stdout


print(run_installed("""
import os
from pathlib import Path
import study_c_api as extension

target = Path(os.environ["STUDY_INSTALL_TARGET"]).resolve()
assert Path(extension.__file__).resolve().parent == target
print(extension.add_nonnegative(20, 55))
print(extension.tuple_item(("C", "Python"), 1))
# 应为 75 和 Python；导入位置必须来自临时安装目标。
"""), end="")

75
Python


## 10 整数转换与错误边界

整数索引协议让某些非 int 对象也能传给 l 或 n。它与把任意值交给 int 转换不同，浮点数即使没有小数部分也不能直接通过本例参数解析。

下面分别观察正常转换、类型错误、负数、输入超出 C long 范围和求和溢出。匹配异常类型后还需继续执行正常调用，检查错误路径没有留下未处理的异常状态。

In [10]:
print(run_installed("""
import study_c_api as extension

class One:
    def __index__(self):
        return 1

print(extension.add_nonnegative(True, False))
print(extension.add_nonnegative(One(), 2))
print(extension.tuple_item(("first", "second"), One()))
limit = extension.LONG_MAX
assert extension.add_nonnegative(limit - 1, 1) == limit
print("C long 上界：", limit)
# 前三项应为 1、3、second；Windows MSVC 的 long 上界为 2147483647。
"""), end="")

1
3
second
C long 上界： 2147483647


### 10.1 按失败发生的位置区分异常

参数数量和关键字实参也属于接口边界。METH\_VARARGS 只接受位置实参，本例的 ll 格式要求恰好两个参数。异常不能仅用“进程没崩溃”来验证，应核对具体类型。

In [11]:
print(run_installed("""
import study_c_api as extension

# 每组只改变一种参数边界，观察异常在哪一层产生。
cases = [
    ((1.0, 0), TypeError),
    ((-1, 0), ValueError),
    ((10**100, 0), OverflowError),
    ((extension.LONG_MAX, 1), OverflowError),
    ((1,), TypeError),
]
for arguments, expected in cases:
    try:
        extension.add_nonnegative(*arguments)
    except expected as error:
        print(type(error).__name__)
    else:
        raise AssertionError("expected exception was not raised")
try:
    extension.add_nonnegative(left=1, right=2)
except TypeError as error:
    print(type(error).__name__)
else:
    raise AssertionError("keywords must be rejected")
assert extension.add_nonnegative(1, 2) == 3
# 顺序应为 TypeError、ValueError、两次 OverflowError、两次 TypeError。
"""), end="")

TypeError
ValueError
OverflowError
OverflowError
TypeError
TypeError


## 11 用对象生命周期观察引用管理

新引用可以指向同一个可变对象。删除原元组后，返回值仍应可用；释放最后一个强引用后，一个没有循环引用的普通对象应能被回收。

弱引用不会单独保持对象存活，所以适合这里观察生命周期。不要把 sys.getrefcount 对小整数的读数当作所有权规则，也不要通过故意删除 Py\_NewRef 制造内存错误来做演示。

In [12]:
print(run_installed("""
import weakref
import study_c_api as extension

class Record:
    pass

record = Record()
record.values = []
observer = weakref.ref(record)
items = (record,)
returned = extension.tuple_item(items, 0)
assert returned is record
# 删除原变量与容器，只保留扩展函数返回的强引用。
del record, items
returned.values.append("alive")
print(returned.values)
assert observer() is returned
# 再释放最后一个强引用，用弱引用观察对象是否已被回收。
del returned
print(observer() is None)
assert observer() is None
assert extension.tuple_item((None,), 0) is None
# 应显示 ['alive'] 和 True；返回引用既没有提前释放，也没有被扩展遗留。
# 元素 None 是合法返回对象，不是 C 层表示失败的 NULL。
"""), end="")

['alive']
True


## 12 检查安装后的公开行为

配套 pytest 检查输入范围、参数个数、关键字、索引转换和异常传播，并重复创建普通对象，检查返回值存活与释放后的回收。它也检查失败的元组访问没有额外保留元素。

测试在新的子进程中调用 pytest.main，返回值作为进程退出状态。关闭插件自动加载与缓存，把测试临时目录放在安装实验目录内。这些用例验证选定行为，不能代替内存分配失败注入、全部引用路径审查或其他平台的测试。

In [13]:
test_directory = str(project_dir / "tests")
print(run_installed(f"""
import pytest

status = pytest.main([
    "-q", "-p", "no:cacheprovider", "--basetemp", "pytest-temp",
    {test_directory!r},
])
raise SystemExit(status)
"""), end="")
# 当前参数组合应形成 27 项测试；返回引用循环 5000 次，错误路径 1000 次。
# 非零退出会由 run_installed 抛出，不能忽略失败测试继续宣称通过。

...........................                                              [100%]
27 passed in 0.20s


## 13 普通 ABI、Stable ABI 与自由线程边界

ABI（application binary interface，应用二进制接口）描述编译后代码之间的接口。普通 CPython 扩展通常需要按 Python 次版本重新构建；同一次版本内的兼容还取决于构建方式和平台条件。

| 标签或名称 | 中文名称／含义 |
| --- | --- |
| cp312 | Python 标签中表示 CPython 3.12；本例 ABI 标签也使用它 |
| win\_amd64 | Windows x64 平台标签 |
| Limited API | 受限 C API，通过 Py\_LIMITED\_API 选择可用 API 子集与最低版本 |
| Stable ABI | 稳定二进制接口，用于符合条件的跨 Python 3 次版本扩展 |
| abi3 | 用于 CPython 稳定 ABI 的标签，不表示跨操作系统或处理器通用 |

选择受限 API 时，应在 Python.h 前定义 Py\_LIMITED\_API，例如 0x030C0000 表示最低 Python 3.12。还需核对实际使用的 API、编译链接选项与 wheel 标签；只改文件名不会使二进制获得兼容性。Windows 的稳定 ABI 扩展应链接 python3.dll，而普通构建使用版本相关的库。本例没有启用受限 API，也没有构建 abi3 wheel。

自由线程是另一项兼容要求。本例在常规 CPython 3.12 下使用 API，没有声明可关闭 GIL。Python 3.14 的扩展文档要求单独检查线程安全、声明支持并构建相应 wheel，且该版本的自由线程构建不支持 Limited API 或 Stable ABI；不能从本章测试推导自由线程安全。

In [14]:
python_tag, abi_tag, platform_tag = wheel_name.removesuffix(".whl").rsplit(
    "-", 3
)[1:]
print(python_tag, abi_tag, platform_tag)
assert (python_tag, abi_tag, platform_tag) == (
    "cp312", "cp312", "win_amd64"
)
assert "#define Py_LIMITED_API" not in c_source
# 预期 cp312 cp312 win_amd64；这不是 py3-none-any 或 abi3 分发文件。
# 当前实际导入通过，不等于已经验证其他 Python、操作系统或架构。

cp312 cp312 win_amd64


## 本章小结

（1）C 指针和 Python 对象有不同的生命周期；借用、新建与被窃取的引用都需要按 API 契约处理。

（2）参数解析格式要匹配 C 变量类型，算术边界要在计算前检查；失败返回值与已设置异常必须一致。

（3）方法表连接调用名称与 C 函数，初始化负责模块创建及各条错误路径的清理。临时构建和子进程导入便于释放文件。

（4）普通 ABI、Stable ABI 和自由线程支持分别有自己的条件，兼容声明需要目标环境中的实际检查。

自查：为什么 PyTuple\_GetItem 的返回值不能未经 Py\_NewRef 就作为本例函数结果交回 Python？为什么不能先计算和，再判断它是否溢出？

## 练习

（1）先预测下面三个输出，再运行核对。说明 bool 的转换策略、返回对象身份，以及 C API 的负索引边界；标准是预测与实际输出一致，并能分别指向参数解析或引用规则。

In [15]:
print(run_installed("""
import study_c_api as extension

items = (["start"],)
returned = extension.tuple_item(items, 0)
returned.append("end")
print(extension.add_nonnegative(True, True))
print(items[0] is returned)
try:
    extension.tuple_item(items, -1)
except IndexError as error:
    print(type(error).__name__)
else:
    raise AssertionError("expected exception was not raised")
"""), end="")
# 先写下预测，再逐项核对；不要修改被调用的 C 源码来匹配预测。

2
True
IndexError


（2）在 run\_installed 的子进程中遍历 0、1、7 的所有两数组合，把扩展结果与 Python 加法比较。再测试 LONG\_MAX 与 0、LONG\_MAX - 1 与 1、LONG\_MAX 与 1、10 的 100 次方与 0、-1 与 0。

核对标准：九组小数值相加一致；前两组边界结果等于 LONG\_MAX，后三组分别抛出 OverflowError、OverflowError、ValueError。每次预期异常都要有未抛出时失败的检查，最后再确认普通调用成功。

In [16]:
exercise_values = [0, 1, 7]
# 在子进程源码中遍历两数组合，并从扩展读取 LONG_MAX 构造边界输入。
# 所有算术比较都用 Python int 作为参照，不在 C 中制造溢出反例。

（3）仿照生命周期实验创建一个含列表属性的普通类，在子进程中重复 100 次取得元组元素、删除原所有者、修改返回对象、释放返回对象。每次另做一次越界访问，并在异常处理结束后释放原元组。

核对标准：返回对象与原元素身份相同，删除原所有者后仍能写入属性；最后释放强引用后弱引用返回 None。越界访问必须为 IndexError，错误路径也不能保留对象。子进程退出为 0，临时安装目录应由 run\_installed 清理。

In [17]:
exercise_rounds = 100
# 使用 weakref.ref 观察普通实例；异常处理结束后再检查对象是否回收。
# 不修改 C 引用管理代码，不用小整数的引用计数充当对象释放证据。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（主体为 3.12） | [扩展接口、头文件、参数与错误约定](https://docs.python.org/3.12/extending/extending.html#a-simple-example)、[错误与异常](https://docs.python.org/3.12/extending/extending.html#intermezzo-errors-and-exceptions)、[方法表与初始化入口](https://docs.python.org/3.12/extending/extending.html#the-module-s-method-table-and-initialization-function)；[参数格式 l、n、O、O!、索引协议及溢出](https://docs.python.org/3.12/c-api/arg.html#numbers)、[对象与格式结束符](https://docs.python.org/3.12/c-api/arg.html#other-objects)；[引用所有权](https://docs.python.org/3.12/c-api/intro.html#reference-counts)、[Py\_NewRef](https://docs.python.org/3.12/c-api/refcounting.html#c.Py_NewRef)、[Py\_DECREF](https://docs.python.org/3.12/c-api/refcounting.html#c.Py_DECREF)、[Py\_XDECREF](https://docs.python.org/3.12/c-api/refcounting.html#c.Py_XDECREF)；[元组取项与负索引](https://docs.python.org/3.12/c-api/tuple.html#c.PyTuple_GetItem)、[设置元组元素与窃取引用](https://docs.python.org/3.12/c-api/tuple.html#c.PyTuple_SetItem)、[C long 转为 Python int](https://docs.python.org/3.12/c-api/long.html#c.PyLong_FromLong)；[模块状态 m\_size](https://docs.python.org/3.12/c-api/module.html#c.PyModuleDef.m_size)、[单阶段初始化](https://docs.python.org/3.12/c-api/module.html#single-phase-initialization)、[多阶段初始化](https://docs.python.org/3.12/c-api/module.html#multi-phase-initialization)、[添加模块属性引用](https://docs.python.org/3.12/c-api/module.html#c.PyModule_AddObjectRef)；[私有堆](https://docs.python.org/3.12/c-api/memory.html#overview)、[bool 与整数](https://docs.python.org/3.12/library/stdtypes.html#boolean-type-bool)、[弱引用生命周期](https://docs.python.org/3.12/library/weakref.html#weakref.ref)；[API 与 ABI 稳定性](https://docs.python.org/3.12/c-api/stable.html#stable-application-binary-interface)、[平台条件](https://docs.python.org/3.12/c-api/stable.html#platform-considerations)、[Windows .pyd 导入](https://docs.python.org/3.12/faq/windows.html#is-a-pyd-file-the-same-as-a-dll)、[Windows 构建](https://docs.python.org/3.12/extending/windows.html#a-cookbook-approach)。自由线程概览单独依据 [3.14 扩展支持声明](https://docs.python.org/3.14/howto/free-threading-extensions.html#module-initialization)、[3.14 构建与稳定 ABI 限制](https://docs.python.org/3.14/howto/free-threading-extensions.html#building-extensions-for-the-free-threaded-build)。辅助操作：[子进程与超时](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[临时目录清理](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[目录复制](https://docs.python.org/3.12/library/shutil.html#shutil.copytree)、[TOML 读取](https://docs.python.org/3.12/library/tomllib.html#tomllib.loads)、[ZIP 读取](https://docs.python.org/3.12/library/zipfile.html#zipfile.ZipFile)、[tar 成员](https://docs.python.org/3.12/library/tarfile.html#tarfile.TarFile.getnames)、[元数据字节解析](https://docs.python.org/3.12/library/email.parser.html#email.parser.BytesParser.parsebytes)、[关闭字节码](https://docs.python.org/3.12/using/cmdline.html#envvar-PYTHONDONTWRITEBYTECODE)。 |
| Microsoft Learn（MSVC 文档） | [指针声明](https://learn.microsoft.com/en-us/cpp/c-language/pointer-declarations?view=msvc-170)、[取地址与解引用](https://learn.microsoft.com/en-us/cpp/c-language/indirection-and-address-of-operators?view=msvc-170)、[局部声明的存储类别](https://learn.microsoft.com/en-us/cpp/c-language/storage-class-specifiers-for-internal-level-declarations?view=msvc-170)、[x64 栈分配](https://learn.microsoft.com/en-us/cpp/build/stack-usage?view=msvc-170#stack-allocation)、[NULL](https://learn.microsoft.com/en-us/cpp/c-runtime-library/null-crt?view=msvc-170)、[C 整数上界](https://learn.microsoft.com/en-us/cpp/c-language/cpp-integer-limits?view=msvc-170)、[UTF-8 编译选项](https://learn.microsoft.com/en-us/cpp/build/reference/utf-8-set-source-and-executable-character-sets-to-utf-8?view=msvc-170)。 |
| CodeQL 官方文档 | [有符号溢出不能在运算后补救](https://codeql.github.com/codeql-query-help/cpp/cpp-signed-overflow-check/#recommendation)，定位 Signed overflow check 的说明与 Recommendation。 |
| GitHub 上的 setuptools 维护者源码 | [setuptools 83.0.0 扩展构建说明](https://github.com/pypa/setuptools/blob/v83.0.0/docs/userguide/ext_modules.rst)，定位 Building Extension Modules 的 setup.py 示例、编译依赖与 Compiler and linker options；本章仅使用 setuptools.Extension 声明普通 ABI 扩展。 |
| PyPA 打包指南与规范 | [Python、ABI 与平台标签](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/#overview)、[ABI 标签](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/#abi-tag)、[平台标签](https://packaging.python.org/en/latest/specifications/platform-compatibility-tags/#platform-tag)，用于解释实际 wheel 文件名。 |
| build 官方文档（1.6.1） | [默认先构建 sdist 再构建 wheel、输出目录、关闭构建隔离](https://build.pypa.io/en/stable/reference/cli.html#python--m-build)。 |
| pip 官方文档 | [目标目录安装](https://pip.pypa.io/en/stable/cli/pip_install/#install-target)、[不查询索引](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-index)、[不安装依赖](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-deps)、[关闭字节码编译](https://pip.pypa.io/en/stable/cli/pip_install/#install-no-compile)。 |
| pytest 官方文档（本章执行 9.1.1） | [预期异常检查](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[从 Python 调用 pytest.main](https://docs.pytest.org/en/stable/how-to/usage.html#calling-pytest-from-python-code)、[禁用插件](https://docs.pytest.org/en/stable/how-to/usage.html#disabling-plugins)、[临时目录位置](https://docs.pytest.org/en/stable/how-to/tmp_path.html#temporary-directory-location-and-retention)。 |